# APUSH FRQ Grader v3
Separate v3 workflow; the v2 notebook remains unchanged and reproducible.

## 1. Runtime and repository

In [2]:
from pathlib import Path
REPO = Path('/content/slm')
%cd /content/slm
%pip install -q -e '.[train]'


/content/slm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for apush-frq-grader-slm (pyproject.toml) ... done


## 2. Drive-backed immutable outputs

In [3]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/slm_v3')
RUN_ROOT.mkdir(parents=True, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Reproduce v2 failure analysis

In [4]:
!python scripts/analyze_v2_for_v3.py


python3: can't open file '/content/slm/scripts/analyze_v2_for_v3.py': [Errno 2] No such file or directory


## 4. Build and audit v3 data
The builder refuses official essays, missing totals/rubric versions, ungrounded feedback, and datasets of 102 rows or fewer.

In [5]:
# Supply reviewed synthetic sources that satisfy every release gate.
# !python scripts/build_v3_dataset.py SOURCE_A.jsonl SOURCE_B.jsonl --target-count 200


## 5. Base 0.5B set1 benchmark

In [6]:
!python scripts/eval_v3.py --model Qwen/Qwen2.5-0.5B-Instruct --model-name Qwen2.5-0.5B-base --output-dir {RUN_ROOT / 'base'}


python3: can't open file '/content/slm/scripts/eval_v3.py': [Errno 2] No such file or directory


## 6. Three-configuration pretraining benchmark

In [7]:
# Pass the base summary emitted above to populate all three configurations.
# !python scripts/benchmark_v3_dev.py --base-summary BASE_SUMMARY.json


## 7. Verify assistant-only labels

In [8]:
!python -m pytest -q tests/test_v3_training.py
!python scripts/audit_v3_tokens.py



no tests ran in 0.00s
ERROR: file or directory not found: tests/test_v3_training.py

python3: can't open file '/content/slm/scripts/audit_v3_tokens.py': [Errno 2] No such file or directory


## 8. Train Qwen 0.5B
Fixed settings: assistant-only loss, 3 epochs, 5% warmup, and 3,072-token context.

In [9]:
DEV05 = f"python scripts/eval_v3.py --model {{checkpoint}} --model-name Qwen2.5-0.5B-v3 --output-dir {RUN_ROOT / 'dev-0.5b'}"
!python scripts/train_v3.py --model Qwen/Qwen2.5-0.5B-Instruct --output {RUN_ROOT / 'qwen-0.5b'} --data artifacts/data/v3/train_chat_v3.jsonl --dev-eval-command "{DEV05}"


python3: can't open file '/content/slm/scripts/train_v3.py': [Errno 2] No such file or directory


## 9. Train Qwen 1.5B

In [10]:
DEV15 = f"python scripts/eval_v3.py --model {{checkpoint}} --model-name Qwen2.5-1.5B-v3 --output-dir {RUN_ROOT / 'dev-1.5b'}"
!python scripts/train_v3.py --model Qwen/Qwen2.5-1.5B-Instruct --output {RUN_ROOT / 'qwen-1.5b'} --data artifacts/data/v3/train_chat_v3.jsonl --dev-eval-command "{DEV15}"


python3: can't open file '/content/slm/scripts/train_v3.py': [Errno 2] No such file or directory


## 10. Evaluate every checkpoint on set1
Run `eval_v3.py` for each saved checkpoint. It reports raw-model and layered-system metrics separately.

In [11]:
# Example:
# !python scripts/eval_v3.py --model CHECKPOINT --model-name CANDIDATE --output-dir {RUN_ROOT / 'dev'}


## 11. Rank checkpoints and freeze a passing candidate

In [12]:
# !python scripts/rank_v3_checkpoints.py SUMMARY_FILES --lock-manifest {RUN_ROOT / 'v3_final_lock.json'}


## 12. Acceptance gate
Do not open set2 unless layered set1 validity is 100%, truncations are zero, MAE is at most 1.5, within-one is at least 60%, and QWK is at least 0.35.

In [13]:
FINAL_EVALUATION = False  # Keep False throughout development.


## 13. Official evaluation split lock
Section 13 defaults to set1. Set2 requires the explicit flag, an exact passing lock manifest, and has a one-run receipt.

In [14]:
MODEL = 'LOCKED_CHECKPOINT'
MODEL_NAME = 'LOCKED_CANDIDATE'
cmd = ['python', 'scripts/eval_v3.py', '--model', MODEL, '--model-name', MODEL_NAME, '--output-dir', str(RUN_ROOT / 'official')]
if FINAL_EVALUATION:
    cmd += ['--final-evaluation', '--lock-manifest', str(RUN_ROOT / 'v3_final_lock.json')]
import subprocess
subprocess.run(cmd, check=True)


CalledProcessError: Command '['python', 'scripts/eval_v3.py', '--model', 'LOCKED_CHECKPOINT', '--model-name', 'LOCKED_CANDIDATE', '--output-dir', '/content/drive/MyDrive/slm_v3/official']' returned non-zero exit status 2.